# Explore the Wildfire STAC Catalog

Round-trip validation for Phase 1: read the self-contained catalog built by
`wildfire_geo_ml.stac_builder.build_catalog` and list every Item with cloud cover
and band asset hrefs.

In [ ]:
from pathlib import Path

import pystac
from pystac.extensions.eo import EOExtension

CATALOG_PATH = Path("../stac/catalog.json")
if not CATALOG_PATH.is_file():
    raise FileNotFoundError(
        f"Catalog not found at {CATALOG_PATH.resolve()}. "
        "Run: uv run python -m wildfire_geo_ml.stac_builder.build_catalog"
    )

catalog = pystac.Catalog.from_file(str(CATALOG_PATH))
items = list(catalog.get_items(recursive=True))
print(f"Catalog: {catalog.id}")
print(f"Items: {len(items)}")

In [ ]:
rows = []
for item in sorted(items, key=lambda i: i.id):
    eo = EOExtension.ext(item)
    cloud = eo.cloud_cover if eo.cloud_cover is not None else "n/a"
    band_assets = [k for k in sorted(item.assets) if k.startswith("B")]
    hrefs = {band: item.assets[band].href for band in band_assets}
    rows.append((item.id, item.datetime, cloud, band_assets, hrefs))
    print(f"{item.id}")
    print(f"  datetime: {item.datetime}")
    print(f"  eo:cloud_cover: {cloud}")
    for band, href in hrefs.items():
        print(f"  {band}: {href}")
    print()

In [ ]:
if items:
    dates = [item.datetime for item in items if item.datetime]
    print(f"Date range: {min(dates)} → {max(dates)}")
    print(f"Total items: {len(items)}")